In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.92)

OUTPUT_ROOT = Path("./one_codon_exhaustive_outputs").resolve()
MAX_ROWS_PER_MODEL = 2_000_000
SAVE_FIGS = True
FIG_DIR = OUTPUT_ROOT / "plots"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ORF_ORDER = [0, 1, 2]
CODON_POSITION_ORDER = [0, 1, 2]
SEQ_TYPE_ORDER = ["coding", "non-coding"]
CONSEQUENCE_ORDER = ["synonymous", "missense"]

POSITION_LABELS = {0: "1st", 1: "2nd", 2: "3rd"}
SEQ_COLORS = {"coding": "#0b4fce", "non-coding": "#c2410c"}
SEQ_PALETTE = {"coding": "#9ec5fe", "non-coding": "#ffb38a"}

print("Using output folder:", OUTPUT_ROOT)
print("Saving figures to:", FIG_DIR)

Using output folder: /storage/bt20d204/ruhaib/Model Evaluation/Mutation Sensitivity/GB sequences/mutation_sensitivity_outputs


In [ ]:
model_dirs = sorted([d for d in OUTPUT_ROOT.iterdir() if d.is_dir()])
model_names = [d.name for d in model_dirs]

records = []
for model_dir in model_dirs:
    for pq_file in sorted(model_dir.glob("*.parquet")):
        records.append({"model": model_dir.name, "file": str(pq_file)})

manifest_df = pd.DataFrame(records)

print("Models found:", model_names)
print("Total parquet files:", len(manifest_df))

if len(manifest_df) == 0:
    raise FileNotFoundError(f"No parquet files found under {OUTPUT_ROOT}")

display(manifest_df.groupby("model").size().rename("parquet_count").reset_index())

Models found: ['DNABERT-2', 'DNABERT-S', 'GPN', 'GROVER', 'GenaLM', 'Hyena-160k', 'Hyena-16k', 'Hyena-1k', 'Hyena-1m', 'Hyena-32k', 'Hyena-450k', 'NT2.5B-1K', 'NT2.5B-MS', 'NT500M', 'PhyloGPN', 'plots']
Total parquet files: 1120


,model,parquet_count
0,DNABERT-2,80
1,DNABERT-S,80
2,GPN,80
3,GROVER,80
4,GenaLM,80
5,Hyena-160k,80
6,Hyena-16k,80
7,Hyena-1k,80
8,Hyena-1m,80
9,Hyena-32k,80


Sample columns:
['model', 'global_sequence_index', 'label', 'sequence_type', 'mutation_percent', 'mutation_instance', 'distance', 'cosine_similarity', 'mutated_sequence', 'mutated_embedding', 'original_embedding']


,model,global_sequence_index,label,sequence_type,mutation_percent,mutation_instance,distance,cosine_similarity,mutated_sequence,mutated_embedding,original_embedding
0,DNABERT-2,37500,0,coding,5,0,4.274433,0.200033,TGGCTGACCTCGATTTAAAGAGACAGCAACTGTCGTGGTCCTGGAA...,"[-0.0412846, -0.18434782, -0.17205137, -0.0522...","[0.050299145, -0.010250635, -0.13859227, -0.15..."
1,DNABERT-2,37500,0,coding,5,1,3.072694,0.533001,TCGCTGGCCTCGATTTAAAGAGACAGAAGCTGTCGGGGTAATGGAA...,"[0.089394964, -0.18095279, -0.2784935, -0.0442...","[0.050299145, -0.010250635, -0.13859227, -0.15..."
2,DNABERT-2,37500,0,coding,5,2,1.033259,0.945611,TGGCTGGCCTCTATTTACAGAGACAGAAGCTGTCGGGGTCCTGGAA...,"[0.044003453, 0.03021047, -0.20713836, -0.1059...","[0.050299145, -0.010250635, -0.13859227, -0.15..."


# per-model figures

In [ ]:
def _safe_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))


def _maybe_sample(df, max_rows=MAX_ROWS_PER_MODEL):
    if len(df) <= max_rows:
        return df
    return df.sample(max_rows, random_state=42).reset_index(drop=True)


def load_model_df(model_name):
    needed_cols = [
        "orf",
        "codon_position",
        "sequence_type",
        "distance",
        "mutation_type_actual",
        "codon_index",
        "mutation_instance",
        "consequence_class",
        "substitution_class",
        "original_base",
        "mutated_base",
    ]

    files = manifest_df.loc[manifest_df["model"] == model_name, "file"].tolist()
    frames = []

    for file_path in files:
        try:
            frame = pd.read_parquet(file_path, columns=[c for c in needed_cols if c in pd.read_parquet(file_path, engine='pyarrow', columns=None).columns])
            frames.append(frame)
        except Exception as exc:
            try:
                # fallback: load without column projection if parquet columns differ
                frame = pd.read_parquet(file_path)
                frames.append(frame)
            except Exception as exc2:
                print(f"[{model_name}] skipped {Path(file_path).name}: {exc2}")

    if len(frames) == 0:
        return pd.DataFrame()

    model_df = pd.concat(frames, ignore_index=True)
    model_df = model_df.dropna(
        subset=["orf", "codon_position", "sequence_type", "distance", "mutation_type_actual", "codon_index"],
    ).copy()

    model_df["orf"] = model_df["orf"].astype(int)
    model_df["codon_position"] = model_df["codon_position"].astype(int)
    model_df["codon_index"] = model_df["codon_index"].astype(int)
    model_df["mutation_instance"] = model_df["mutation_instance"].astype(int)
    model_df["distance"] = model_df["distance"].astype(float)
    model_df["sequence_type"] = pd.Categorical(
        model_df["sequence_type"], categories=SEQ_TYPE_ORDER, ordered=True
    )
    model_df["mutation_type_actual"] = pd.Categorical(
        model_df["mutation_type_actual"],
        categories=["synonymous", "missense", "nonsense", "stop_lost"],
        ordered=False,
    )
    model_df["consequence_class"] = pd.Categorical(
        model_df["consequence_class"], categories=CONSEQUENCE_ORDER, ordered=False
    )

    # substitution_class may be missing in older parquet exports; try to infer if bases are present
    if "substitution_class" not in model_df.columns and set(["original_base", "mutated_base"]).issubset(model_df.columns):
        def _is_transition(row):
            a = row["original_base"].upper()
            b = row["mutated_base"].upper()
            return (a == b) is False and ( (a in "AG" and b in "AG") or (a in "CT" and b in "CT") )
        model_df["substitution_class"] = model_df.apply(lambda r: "transition" if _is_transition(r) else "transversion", axis=1)

    return _maybe_sample(model_df)


def _legend_handles():
    return [
        Patch(facecolor=SEQ_PALETTE["coding"], edgecolor="0.35", label="coding boxes"),
        Patch(facecolor=SEQ_PALETTE["non-coding"], edgecolor="0.35", label="non-coding boxes"),
        Line2D([0], [0], color=SEQ_COLORS["coding"], marker="o", linewidth=2.2, label="coding median"),
        Line2D([0], [0], color=SEQ_COLORS["non-coding"], marker="o", linewidth=2.2, label="non-coding median"),
    ]


def _save_fig(fig, *parts):
    if not SAVE_FIGS:
        return None
    out_path = FIG_DIR.joinpath(*[_safe_name(part) for part in parts])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    print("Saved:", out_path)
    return out_path


def _seq_type_color_from_label(label):
    return SEQ_PALETTE["coding"] if str(label).split("|")[-1].strip() == "coding" else SEQ_PALETTE["non-coding"]


def _plot_boxes_with_medians(ax, df, metric_col, order, title, y_label, rotate=0):
    if len(order) == 0:
        ax.set_axis_off()
        return

    plot_df = df.copy()

    sns.boxplot(
        data=plot_df,
        x="panel_label",
        y=metric_col,
        order=order,
        palette={label: _seq_type_color_from_label(label) for label in order},
        showfliers=False,
        ax=ax,
    )

    x_lookup = {label: idx for idx, label in enumerate(order)}
    for seq_type in SEQ_TYPE_ORDER:
        xs = []
        ys = []
        for label in order:
            if not label.endswith(f"| {seq_type}"):
                continue
            row = plot_df[plot_df["panel_label"] == label]
            if len(row) == 0:
                continue
            xs.append(x_lookup[label])
            ys.append(float(row[metric_col].median()))
        if len(xs) > 0:
            ax.plot(xs, ys, color=SEQ_COLORS[seq_type], marker="o", linewidth=2.2)

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(y_label)
    ax.set_xticklabels(order, rotation=rotate)
    ax.legend(handles=_legend_handles(), title="Legend", frameon=True, loc="best", fontsize=8, title_fontsize=9)
    sns.despine(ax=ax)


def _plot_orf_summary_panel(ax, df, metric_col, title, y_label):
    panel_df = df.copy()
    panel_df["panel_label"] = (
        panel_df["codon_position"].map(POSITION_LABELS) + " | " + panel_df["sequence_type"].astype(str)
    )
    order = [
        f"{POSITION_LABELS[pos]} | {seq_type}"
        for pos in CODON_POSITION_ORDER
        for seq_type in SEQ_TYPE_ORDER
    ]
    _plot_boxes_with_medians(
        ax=ax,
        df=panel_df,
        metric_col=metric_col,
        order=order,
        title=title,
        y_label=y_label,
        rotate=0,
    )
    for boundary in [1.5, 3.5]:
        ax.axvline(boundary, color="0.85", linewidth=1)


def _plot_syn_mis_combined_panel(ax, df, metric_col, title, y_label):
    panel_df = df.copy()
    panel_df["prefix"] = panel_df["mutation_type_actual"].map({"synonymous": "Syn", "missense": "Mis"})
    panel_df["panel_label"] = (
        panel_df["prefix"]
        + " | "
        + panel_df["codon_position"].map(POSITION_LABELS)
        + " | "
        + panel_df["sequence_type"].astype(str)
    )

    order = []
    # Build order dynamically only for non-empty cells (keeps gaps away)
    for mutation_type in ["synonymous", "missense"]:
        type_df = panel_df[panel_df["mutation_type_actual"] == mutation_type]
        if len(type_df) == 0:
            continue
        prefix = "Syn" if mutation_type == "synonymous" else "Mis"
        for pos in CODON_POSITION_ORDER:
            pos_df = type_df[type_df["codon_position"] == pos]
            if len(pos_df) == 0:
                continue
            for seq_type in SEQ_TYPE_ORDER:
                cell_df = pos_df[pos_df["sequence_type"] == seq_type]
                if len(cell_df) == 0:
                    continue
                order.append(f"{prefix} | {POSITION_LABELS[pos]} | {seq_type}")

    _plot_boxes_with_medians(
        ax=ax,
        df=panel_df,
        metric_col=metric_col,
        order=order,
        title=title,
        y_label=y_label,
        rotate=30,
    )

    # Draw a bolder separator between Synonymous and Missense groups (if both present)
    if len(order) > 0:
        n_syn = sum(1 for lbl in order if str(lbl).strip().startswith("Syn"))
        if n_syn > 0 and n_syn < len(order):
            sep_x = n_syn - 0.5
            ax.axvline(sep_x, color="0.7", linewidth=2.2)

    # Replace verbose xtick labels (which include sequence_type) with just the codon position label
    if len(order) > 0:
        short_labels = [lbl.split("|")[1].strip() if "|" in str(lbl) else str(lbl) for lbl in order]
        ax.set_xticklabels(short_labels, rotation=30)


def _plot_codon_progress_panel(ax, df, metric_col, title, y_label):
    df = df.copy()
    df["codon_index_display"] = df["codon_index"].astype(int) + 1
    max_index = int(df["codon_index_display"].max())
    order = list(range(1, max_index + 1))

    sns.boxplot(
        data=df,
        x="codon_index_display",
        y=metric_col,
        hue="sequence_type",
        order=order,
        palette=SEQ_PALETTE,
        showfliers=False,
        ax=ax,
    )

    medians = df.groupby(["codon_index_display", "sequence_type"], as_index=False)[metric_col].median()
    for seq_type in SEQ_TYPE_ORDER:
        sub = medians[medians["sequence_type"] == seq_type].sort_values("codon_index_display")
        if len(sub) > 0:
            ax.plot(
                sub["codon_index_display"],
                sub[metric_col],
                color=SEQ_COLORS[seq_type],
                marker="o",
                linewidth=2.0,
            )

    ax.set_title(title)
    ax.set_xlabel("Codon index within ORF")
    ax.set_ylabel(y_label)
    ax.set_xticks(order[::5] if len(order) > 20 else order)
    ax.set_xticklabels([str(x) for x in (order[::5] if len(order) > 20 else order)], rotation=0)
    ax.legend(handles=_legend_handles(), title="Legend", frameon=True, loc="best", fontsize=8, title_fontsize=9)
    sns.despine(ax=ax)


def plot_all_for_model(model_name):
    model_df = load_model_df(model_name)
    if len(model_df) == 0:
        print(f"[{model_name}] no rows found.")
        return

    print(f"[{model_name}] rows loaded: {len(model_df):,}")

    # Family 1: all mutations, 3 ORFs side by side.
    fig, axes = plt.subplots(1, 3, figsize=(24, 5.3), sharey=True)
    for orf, ax in zip(ORF_ORDER, axes):
        orf_df = model_df[model_df["orf"] == orf].copy()
        if len(orf_df) == 0:
            ax.set_axis_off()
            continue
        _plot_orf_summary_panel(
            ax=ax,
            df=orf_df,
            metric_col="distance",
            title=f"ORF {orf}",
            y_label="Euclidean distance" if orf == 0 else "",
        )
    fig.suptitle(f"{model_name} | All mutations | ORF comparison", fontsize=14, y=1.02)
    plt.tight_layout()
    _save_fig(fig, model_name, "family_1_all_mutations_orf_triptych.png")
    plt.show()

    # Family 2: synonymous and missense merged in the same panel, with synonymous on the left.
    fig, axes = plt.subplots(1, 3, figsize=(24, 5.6), sharey=True)
    for orf, ax in zip(ORF_ORDER, axes):
        orf_df = model_df[model_df["orf"] == orf].copy()
        if len(orf_df) == 0:
            ax.set_axis_off()
            continue
        _plot_syn_mis_combined_panel(
            ax=ax,
            df=orf_df,
            metric_col="distance",
            title=f"ORF {orf}",
            y_label="Euclidean distance" if orf == 0 else "",
        )
    fig.suptitle(f"{model_name} | Synonymous vs missense | ORF comparison", fontsize=14, y=1.02)
    plt.tight_layout()
    _save_fig(fig, model_name, "family_2_synonymous_missense_orf_triptych.png")
    plt.show()


## nucleotide position based coding/non coding comparison

In [ ]:
for model_name in model_names:
    model_df = load_model_df(model_name)
    if len(model_df) == 0:
        print(f"[{model_name}] no rows found.")
        continue

    print(f"[{model_name}] rows loaded: {len(model_df):,}")

    fig, axes = plt.subplots(1, 3, figsize=(24, 5.3), sharey=True)
    for orf, ax in zip(ORF_ORDER, axes):
        orf_df = model_df[model_df["orf"] == orf].copy()
        if len(orf_df) == 0:
            ax.set_axis_off()
            continue

        sns.boxplot(
            data=orf_df,
            x="codon_position",
            y="distance",
            hue="sequence_type",
            order=CODON_POSITION_ORDER,
            hue_order=SEQ_TYPE_ORDER,
            palette=SEQ_PALETTE,
            showfliers=False,
            ax=ax,
        )

        medians = orf_df.groupby(["codon_position", "sequence_type"], as_index=False)["distance"].median()
        x_positions = {codon_pos: idx for idx, codon_pos in enumerate(CODON_POSITION_ORDER)}
        offsets = {"coding": -0.18, "non-coding": 0.18}
        for seq_type in SEQ_TYPE_ORDER:
            sub = medians[medians["sequence_type"] == seq_type].sort_values("codon_position")
            if len(sub) > 0:
                xs = [x_positions[codon_pos] + offsets[seq_type] for codon_pos in sub["codon_position"]]
                ax.plot(
                    xs,
                    sub["distance"],
                    color=SEQ_COLORS[seq_type],
                    marker="o",
                    linewidth=2.0,
                )

        for boundary in (0.5, 1.5):
            ax.axvline(boundary, color="0.85", linewidth=1)

        ax.set_title(f"ORF {orf}")
        ax.set_xlabel("Codon position")
        ax.set_ylabel("Euclidean distance" if orf == 0 else "")
        ax.set_xticklabels([POSITION_LABELS[pos] for pos in CODON_POSITION_ORDER])
        legend = ax.legend(title="Legend", frameon=True, loc="best", fontsize=8, title_fontsize=9)
        if legend is not None:
            legend._legend_box.sep = 4
        sns.despine(ax=ax)

    fig.suptitle(f"{model_name} | All mutations | ORF comparison", fontsize=14, y=1.02)
    plt.tight_layout()
    _save_fig(fig, model_name, "family_1_all_mutations_orf_triptych.png")
    plt.show()

## synonymous v/s missense comparison

In [ ]:
for model_name in model_names:
    model_df = load_model_df(model_name)
    if len(model_df) == 0:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(24, 5.6), sharey=True)
    for orf, ax in zip(ORF_ORDER, axes):
        orf_df = model_df[model_df["orf"] == orf].copy()
        if len(orf_df) == 0:
            ax.set_axis_off()
            continue

        _plot_syn_mis_combined_panel(
            ax=ax,
            df=orf_df,
            metric_col="distance",
            title=f"ORF {orf}",
            y_label="Euclidean distance" if orf == 0 else "",
        )

    fig.suptitle(f"{model_name} | Synonymous vs missense | ORF comparison", fontsize=14, y=1.02)
    plt.tight_layout()
    _save_fig(fig, model_name, "family_2_synonymous_missense_orf_triptych.png")
    plt.show()

## comparison across 'codon' positions

## per-model ORF detection via synonymous mutations

In [ ]:
for model_name in model_names:
    model_df = load_model_df(model_name)
    if len(model_df) == 0:
        continue

    for orf in ORF_ORDER:
        orf_df = model_df[model_df["orf"] == orf].copy()
        if len(orf_df) == 0:
            continue

        fig, axes = plt.subplots(1, 3, figsize=(28, 5.5), sharey=True)
        for codon_pos, ax in zip(CODON_POSITION_ORDER, axes):
            pos_df = orf_df[orf_df["codon_position"] == codon_pos].copy()
            if len(pos_df) == 0:
                ax.set_axis_off()
                continue

            _plot_codon_progress_panel(
                ax=ax,
                df=pos_df,
                metric_col="distance",
                title=f"{POSITION_LABELS[codon_pos]} codon position",
                y_label="Euclidean distance" if codon_pos == 0 else "",
            )
            if codon_pos != 0:
                ax.set_ylabel("")

        fig.suptitle(f"{model_name} | ORF {orf} | Codon-index progression", fontsize=14, y=1.02)
        plt.tight_layout()
        _save_fig(fig, model_name, f"orf_{orf}", "family_3_codon_index_progression.png")
        plt.show()


# cross model analysis

hypothesis: the best ORF for the coding sequences should be the one that shows the lowest distances for the synonymous mutations across the 3 ORFs, as that would be the actual codon distribution/true frame. I'll compare boxplots across the 3 positions for the first, second and third ORF altogether in one plot, for each model; If all the models show the least distances for the same ORF, then it would indicate that that's the actual ORF - i think that would be a cool demonstration - to show that these models can understand which is the correct ORF too??

In [ ]:
def plot_cross_model_orf_detection():
    """
    Cross-model ORF detection via synonymous mutations.
    
    For each codon position (1st/2nd/3rd), create a plot with:
    - X-axis: model names
    - Grouped boxes: ORF 0, ORF 1, ORF 2 for each model
    - Y-axis: embedding distance for synonymous mutations
    - Separate lines for coding and non-coding
    
    The hypothesis: the true ORF should show the smallest distances in coding.
    """
    fig, axes = plt.subplots(1, 3, figsize=(28, 6.5), sharey=True)
    
    for codon_pos, ax in zip(CODON_POSITION_ORDER, axes):
        plot_data = []
        
        for model_name in model_names:
            model_df = load_model_df(model_name)
            if len(model_df) == 0:
                continue
            
            syn_df = model_df[
                (model_df["mutation_type_actual"] == "synonymous")
                & (model_df["codon_position"] == codon_pos)
            ].copy()
            
            if len(syn_df) == 0:
                continue
            
            for orf in ORF_ORDER:
                orf_syn_df = syn_df[syn_df["orf"] == orf]
                if len(orf_syn_df) == 0:
                    continue
                
                for seq_type in SEQ_TYPE_ORDER:
                    type_df = orf_syn_df[orf_syn_df["sequence_type"] == seq_type]
                    if len(type_df) == 0:
                        continue
                    
                    plot_data.append({
                        "model": model_name,
                        "orf": orf,
                        "sequence_type": seq_type,
                        "distance": type_df["distance"].values,
                    })
        
        if len(plot_data) == 0:
            ax.set_axis_off()
            continue
        
        df_plot = pd.DataFrame()
        for row in plot_data:
            for dist in row["distance"]:
                df_plot = pd.concat([
                    df_plot,
                    pd.DataFrame({
                        "model": [row["model"]],
                        "orf": [row["orf"]],
                        "sequence_type": [row["sequence_type"]],
                        "distance": [dist],
                    })
                ], ignore_index=True)
        
        df_plot["orf_label"] = "ORF " + df_plot["orf"].astype(str)
        df_plot = df_plot.sort_values(["model", "orf"])
        
        order = sorted(df_plot["model"].unique())
        
        sns.boxplot(
            data=df_plot,
            x="model",
            y="distance",
            hue="orf_label",
            hue_order=["ORF 0", "ORF 1", "ORF 2"],
            order=order,
            palette="Set2",
            showfliers=False,
            ax=ax,
        )
        
        medians = df_plot.groupby(["model", "orf"], as_index=False)["distance"].median()
        x_lookup = {model: idx for idx, model in enumerate(order)}
        
        orf_colors_median = {"ORF 0": "#1f77b4", "ORF 1": "#ff7f0e", "ORF 2": "#2ca02c"}
        
        for orf in ORF_ORDER:
            orf_label = f"ORF {orf}"
            sub = medians[medians["orf"] == orf].sort_values("model")
            sub = sub[sub["model"].isin(order)]
            if len(sub) > 0:
                xs = [x_lookup[model] for model in sub["model"]]
                ys = sub["distance"].values
                ax.plot(xs, ys, color=orf_colors_median[orf_label], marker="o", linewidth=2.2)
        
        ax.set_title(f"{POSITION_LABELS[codon_pos]} codon position")
        ax.set_xlabel("Model" if codon_pos == 1 else "")
        ax.set_ylabel("Euclidean distance" if codon_pos == 0 else "")
        ax.set_xticklabels(order, rotation=45, ha="right")
        sns.despine(ax=ax)
        if codon_pos == 2:
            ax.legend(title="ORF", loc="best")
    
    fig.suptitle("Cross-model ORF detection: Synonymous mutations by position", fontsize=14, y=0.98)
    plt.tight_layout()
    _save_fig(fig, "cross_model_orf_detection_3panels.png")
    plt.show()

In [ ]:
def plot_cross_model_mutation_gap():
    """
    Cross-model mutation-gap comparison.
    
    For each codon position (1st/2nd/3rd), create a plot with:
    - X-axis: model names
    - 2 boxplots per model: one for coding, one for non-coding
    - Y-axis: embedding distance for ALL mutation types (not just synonymous)
    
    This shows the absolute gap between coding and non-coding perturbations,
    independent of mutation consequence. Models sensitive to codon structure
    should show larger gaps.
    """
    fig, axes = plt.subplots(1, 3, figsize=(28, 6.5), sharey=True)
    
    for codon_pos, ax in zip(CODON_POSITION_ORDER, axes):
        plot_data = []
        
        for model_name in model_names:
            model_df = load_model_df(model_name)
            if len(model_df) == 0:
                continue
            
            pos_df = model_df[model_df["codon_position"] == codon_pos].copy()
            
            if len(pos_df) == 0:
                continue
            
            for seq_type in SEQ_TYPE_ORDER:
                type_df = pos_df[pos_df["sequence_type"] == seq_type]
                if len(type_df) == 0:
                    continue
                
                for dist in type_df["distance"].values:
                    plot_data.append({
                        "model": model_name,
                        "sequence_type": seq_type,
                        "distance": dist,
                    })
        
        if len(plot_data) == 0:
            ax.set_axis_off()
            continue
        
        df_plot = pd.DataFrame(plot_data)
        df_plot = df_plot.sort_values(["model", "sequence_type"])
        
        order = sorted(df_plot["model"].unique())
        
        sns.boxplot(
            data=df_plot,
            x="model",
            y="distance",
            hue="sequence_type",
            order=order,
            palette=SEQ_PALETTE,
            showfliers=False,
            ax=ax,
        )
        
        medians = df_plot.groupby(["model", "sequence_type"], as_index=False)["distance"].median()
        x_lookup = {model: idx for idx, model in enumerate(order)}
        
        for seq_type in SEQ_TYPE_ORDER:
            sub = medians[medians["sequence_type"] == seq_type].sort_values("model")
            sub = sub[sub["model"].isin(order)]
            if len(sub) > 0:
                xs = [x_lookup[model] for model in sub["model"]]
                ys = sub["distance"].values
                ax.plot(xs, ys, color=SEQ_COLORS[seq_type], marker="o", linewidth=2.2)
        
        ax.set_title(f"{POSITION_LABELS[codon_pos]} codon position")
        ax.set_xlabel("Model" if codon_pos == 1 else "")
        ax.set_ylabel("Euclidean distance (all mutations)" if codon_pos == 0 else "")
        ax.set_xticklabels(order, rotation=45, ha="right")
        sns.despine(ax=ax)
        if codon_pos == 2:
            ax.legend(title="Sequence Type", loc="best")
    
    fig.suptitle("Cross-model coding/non-coding gap: All mutations by position", fontsize=14, y=0.98)
    plt.tight_layout()
    _save_fig(fig, "cross_model_mutation_gap_3panels.png")
    plt.show()

In [ ]:
def analyze_cross_model_conclusions():
    """
    Summarize cross-model findings:
    1. Which ORF is most consistently identified across models via synonymous mutations?
    2. Does that ORF show the clearest coding/non-coding gap?
    3. Which models show the strongest codon-structure signal?
    """
    print("=" * 80)
    print("CROSS-MODEL CODON-DEGENERACY ANALYSIS")
    print("=" * 80)
    
    orf_votes = {orf: {codon_pos: [] for codon_pos in CODON_POSITION_ORDER} for orf in ORF_ORDER}
    model_gap_scores = []
    
    for model_name in model_names:
        model_df = load_model_df(model_name)
        if len(model_df) == 0:
            continue
        
        syn_df = model_df[model_df["mutation_type_actual"] == "synonymous"].copy()
        if len(syn_df) == 0:
            continue
        
        model_gaps = []
        
        for codon_pos in CODON_POSITION_ORDER:
            pos_df = syn_df[syn_df["codon_position"] == codon_pos]
            if len(pos_df) == 0:
                continue
            
            orf_stats = {}
            for orf in ORF_ORDER:
                orf_df = pos_df[pos_df["orf"] == orf]
                coding_df = orf_df[orf_df["sequence_type"] == "coding"]
                if len(coding_df) > 0:
                    orf_stats[orf] = float(coding_df["distance"].median())
            
            if len(orf_stats) == 0:
                continue
            
            best_orf = min(orf_stats, key=orf_stats.get)
            orf_votes[best_orf][codon_pos].append(model_name)
            
            noncoding_df = pos_df[pos_df["sequence_type"] == "non-coding"]
            coding_median = float(coding_df["distance"].median()) if len(coding_df) > 0 else None
            noncoding_median = float(noncoding_df["distance"].median()) if len(noncoding_df) > 0 else None
            
            if coding_median is not None and noncoding_median is not None:
                gap = noncoding_median - coding_median
                model_gaps.append(gap)
        
        if len(model_gaps) > 0:
            model_gap_scores.append({
                "model": model_name,
                "mean_gap": float(np.mean(model_gaps)),
                "median_gap": float(np.median(model_gaps)),
            })
    
    print("\n1. ORF IDENTIFICATION (Synonymous mutations - Coding sequences should show smallest distance)")
    print("-" * 80)
    for orf in ORF_ORDER:
        print(f"\nORF {orf} voted best by models at codon positions:")
        for codon_pos in CODON_POSITION_ORDER:
            voters = orf_votes[orf][codon_pos]
            if len(voters) > 0:
                print(f"  Position {POSITION_LABELS[codon_pos]}: {', '.join(voters)}")
            else:
                print(f"  Position {POSITION_LABELS[codon_pos]}: (no votes)")
    
    print("\n2. MODEL SENSITIVITY RANKING (Coding/non-coding gap for synonymous mutations)")
    print("-" * 80)
    if len(model_gap_scores) > 0:
        gap_df = pd.DataFrame(model_gap_scores).sort_values("median_gap", ascending=False)
        print("\nMedian gap (non-coding - coding distance):")
        for idx, row in gap_df.iterrows():
            print(f"  {row['model']:20s} : {row['median_gap']:8.4f}")
    
    print("\n3. INTERPRETATION")
    print("-" * 80)
    print("""
    Strong codon-structure learning signal would show:
    - Convergence: Multiple models identify the SAME ORF as best across positions.
    - Asymmetry: That ORF shows large coding/non-coding gap (codon degeneracy buffering).
    - Ranking: Models with highest gap scores are most sensitive to codon structure.
    
    Weak signal would show:
    - Inconsistent ORF votes across models.
    - No clear coding/non-coding separation.
    - Small gap scores across the board.
    """)
    
    return {
        "orf_votes": orf_votes,
        "model_gaps": gap_df if len(model_gap_scores) > 0 else pd.DataFrame(),
    }

In [ ]:
def plot_transition_transversion_cross_model(by_position=False):
    """
    Compare transition vs transversion effects across models.

    If `by_position` is False (default): produces two side-by-side bar plots (coding, non-coding)
    with median distance per model and substitution class (transition/transversion).

    If `by_position` is True: includes codon_position as an additional grouping (slower).
    """
    rows = []

    for model_name in model_names:
        model_df = load_model_df(model_name)
        if len(model_df) == 0:
            continue

        # Ensure we have substitution class information (attempt inference if needed)
        if "substitution_class" not in model_df.columns and set(["original_base", "mutated_base"]).issubset(model_df.columns):
            def _is_transition_pair(a, b):
                a = a.upper(); b = b.upper()
                return (a in "AG" and b in "AG") or (a in "CT" and b in "CT")
            model_df["substitution_class"] = model_df.apply(lambda r: "transition" if _is_transition_pair(r["original_base"], r["mutated_base"]) else "transversion", axis=1)

        df = model_df[model_df["substitution_class"].isin(["transition", "transversion"])].copy()
        if len(df) == 0:
            continue

        if by_position:
            grp = df.groupby(["model", "substitution_class", "sequence_type", "codon_position"], as_index=False)["distance"].median()
            for _, r in grp.iterrows():
                rows.append({
                    "model": model_name,
                    "substitution_class": r["substitution_class"],
                    "sequence_type": r["sequence_type"],
                    "codon_position": int(r["codon_position"]),
                    "median_distance": float(r["distance"]),
                })
        else:
            grp = df.groupby(["substitution_class", "sequence_type"], as_index=False)["distance"].median()
            for _, r in grp.iterrows():
                rows.append({
                    "model": model_name,
                    "substitution_class": r["substitution_class"],
                    "sequence_type": r["sequence_type"],
                    "median_distance": float(r["distance"]),
                })

    df_plot = pd.DataFrame(rows)
    if df_plot.empty:
        print("No transition/transversion data available across models.")
        return df_plot

    if not by_position:
        fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
        for i, seq_type in enumerate(SEQ_TYPE_ORDER):
            sub = df_plot[df_plot["sequence_type"] == seq_type]
            if sub.empty:
                axes[i].set_axis_off()
                continue
            order = sorted(sub["model"].unique())
            sns.barplot(data=sub, x="model", y="median_distance", hue="substitution_class", order=order, ax=axes[i])
            axes[i].set_title(f"{seq_type} (median distance)")
            axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha="right")
            axes[i].set_ylabel("Euclidean distance" if i == 0 else "")
            axes[i].legend(title="Substitution", fontsize=8, title_fontsize=9)
            sns.despine(ax=axes[i])
        plt.tight_layout()
        _save_fig(fig, "cross_model_transition_vs_transversion.png")
        plt.show()
    else:
        fig, ax = plt.subplots(1, 1, figsize=(14, 6))
        sns.boxplot(data=df_plot, x="codon_position", y="median_distance", hue="substitution_class", palette="Set1", ax=ax)
        ax.set_title("Transition vs Transversion (median distances by codon position)")
        ax.set_xlabel("Codon position")
        ax.set_xticklabels([POSITION_LABELS.get(int(x), str(x)) for x in sorted(df_plot["codon_position"].unique())])
        ax.legend(title="Substitution", fontsize=8, title_fontsize=9)
        plt.tight_layout()
        _save_fig(fig, "cross_model_transition_vs_transversion_by_position.png")
        plt.show()

    return df_plot

In [ ]:
plot_cross_model_orf_detection()
plot_cross_model_mutation_gap()
conclusions = analyze_cross_model_conclusions()